# 05 — Evaluation, Visualization & Summary
Full evaluation and all publication-quality plots.

In [ ]:
import sys
sys.path.insert(0, "..")
from src.evaluation import evaluate_all, get_uplift_curve, calibration_by_decile
from src.viz import (
    plot_uplift_curves, plot_qini_curves, plot_cate_distribution,
    plot_calibration, plot_metrics_bar, plot_refutation_table
)
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_parquet("../data/simulation_observational.parquet")
feature_cols = [c for c in df.columns if c not in ["treatment", "outcome", "tau_true", "propensity"]]

with open("../results/meta_learner_results.pkl", "rb") as f:
    meta_results = pickle.load(f)

import os
dml_path = "../results/dml_tau_hat.npy"
tau_dml = np.load(dml_path) if os.path.exists(dml_path) else None

tau_hats = {name: res["tau_hat"] for name, res in meta_results.items()}
if tau_dml is not None:
    tau_hats["DoubleML"] = tau_dml

T = df["treatment"].values
Y = df["outcome"].values

## All metrics

In [ ]:
metrics = evaluate_all(df, tau_hats)
metrics

## Uplift curves (all models)

In [ ]:
uplift_dfs = {name: get_uplift_curve(tau, T, Y) for name, tau in tau_hats.items()}
fig = plot_uplift_curves(uplift_dfs, save=True)
fig.show()

## Qini curves

In [ ]:
tau_rand = np.random.default_rng(0).uniform(0, 1, size=len(T))
random_uplift = get_uplift_curve(tau_rand, T, Y)
fig = plot_qini_curves(uplift_dfs, random_uplift=random_uplift, save=True)
fig.show()

## Calibration plots

In [ ]:
model_subset = ["S", "T", "X", "R", "DoubleML"]
cal_dfs = {
    name: calibration_by_decile(tau, T, Y, n_deciles=10)
    for name, tau in tau_hats.items()
    if name in model_subset
}
fig = plot_calibration(cal_dfs, save=True)
fig.show()

## AUUC bar chart

In [ ]:
fig = plot_metrics_bar(metrics, metric="AUUC", save=True)
fig.show()

## Qini bar chart

In [ ]:
fig = plot_metrics_bar(metrics, metric="Qini", save=True)
fig.show()

## PEHE bar chart

In [ ]:
fig = plot_metrics_bar(metrics.dropna(), metric="PEHE", save=True)
fig.show()

## Summary table (paper-ready with gradient styling)

In [ ]:
from IPython.display import display
display(metrics.style
    .background_gradient(cmap="RdYlGn", axis=0, subset=["AUUC", "Qini"])
    .background_gradient(cmap="RdYlGn_r", axis=0, subset=["PEHE"]))